In [ ]:
import os
import pandas as pd
import numpy as np

#### Constants

In [ ]:
str_bucket = os.getcwd().split('/')[4].replace('_','-')
print(f'Bucket: {str_bucket}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# main pitch types to keep
LIST_MAIN_TYPES = ['FF', 'SL', 'SI', 'FT', 'CH', 'CU', 'FC', 'FS']

#### Import Data

In [ ]:
str_uri = f's3://{str_bucket}/00_data_collection/pitches'
df = pd.read_csv(str_uri)
print(f'Raw shape: {df.shape}')
df.head()

#### Filter to Main Pitch Types

In [ ]:
# drop rows with missing pitch_type
df = df[df['pitch_type'].notna()].copy()
print(f'After dropping missing pitch_type: {df.shape}')

# keep only main pitch types (drops IN, PO, AB, UN, KN, EP, SC, FO, FA)
df = df[df['pitch_type'].isin(LIST_MAIN_TYPES)].copy()
print(f'After filtering to main types: {df.shape}')
print(f'\nPitch type distribution:')
print(df['pitch_type'].value_counts())

#### Sort Chronologically

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['date', 'game_pk', 'at_bat_num', 'pcount_at_bat']).reset_index(drop=True)
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Unique dates: {df["date"].nunique()}')
print(f'Unique games: {df["game_pk"].nunique()}')

#### Out-of-Time Split

Split by date to simulate real-world deployment (predict future pitches from past data).
- **Train**: ~70% (earliest games)
- **Validation**: ~15% (middle games, for hyperparameter tuning)
- **Test**: ~15% (latest games, final evaluation)

In [ ]:
# get unique dates sorted
arr_dates = np.sort(df['date'].unique())
n_dates = len(arr_dates)
print(f'Total unique dates: {n_dates}')

# split points
int_train_end = int(n_dates * 0.70)
int_valid_end = int(n_dates * 0.85)

dt_train_cutoff = arr_dates[int_train_end]
dt_valid_cutoff = arr_dates[int_valid_end]

print(f'Train: start to {dt_train_cutoff} ({int_train_end} dates)')
print(f'Valid: {dt_train_cutoff} to {dt_valid_cutoff} ({int_valid_end - int_train_end} dates)')
print(f'Test:  {dt_valid_cutoff} to end ({n_dates - int_valid_end} dates)')

In [ ]:
df_train = df[df['date'] < dt_train_cutoff].copy()
df_valid = df[(df['date'] >= dt_train_cutoff) & (df['date'] < dt_valid_cutoff)].copy()
df_test = df[df['date'] >= dt_valid_cutoff].copy()

print(f'Train: {df_train.shape[0]:,} rows ({df_train.shape[0]/len(df)*100:.1f}%) | {df_train["date"].min()} to {df_train["date"].max()}')
print(f'Valid: {df_valid.shape[0]:,} rows ({df_valid.shape[0]/len(df)*100:.1f}%) | {df_valid["date"].min()} to {df_valid["date"].max()}')
print(f'Test:  {df_test.shape[0]:,} rows ({df_test.shape[0]/len(df)*100:.1f}%) | {df_test["date"].min()} to {df_test["date"].max()}')

In [ ]:
# verify pitch type distribution is similar across splits
df_split_dist = pd.DataFrame({
    'train': df_train['pitch_type'].value_counts(normalize=True) * 100,
    'valid': df_valid['pitch_type'].value_counts(normalize=True) * 100,
    'test': df_test['pitch_type'].value_counts(normalize=True) * 100,
}).round(1)
print('Pitch type distribution (%) across splits:')
df_split_dist

#### Save Splits to S3

In [ ]:
for str_name, df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    str_s3_path = f's3://{str_bucket}/{str_task}/{str_name}.csv'
    df_split.to_csv(str_s3_path, index=False)
    print(f'Saved {str_name}.csv to {str_s3_path} ({df_split.shape})')